# 09 — TF-activity inference (decoupleR 2.x + CollecTRI)

**Question:** who controls the coupled UPR–chemokine module? AR (A06's RF-ranked
mediator) is transcriptionally absent in this organoid (verified: 0–16 reads/locus,
STAR ReadsPerGene 0/0/0, vs TMPRSS2 14–35k). So AR cannot be the controller here.
This notebook nominates candidate controllers from the **expressed** transcriptome.

**Approach:** ULM TF-activity on the DESeq2 Wald stats per contrast, against CollecTRI,
with regulons **restricted to genes expressed in A04** (this is the methodological fix
for the AR trap — no topology-only nodes). Candidate shared controller = a TF whose
activity goes **down with butyrate in Cd** *and* **up with infection at 0 mM**.

Kernel: `decoupler` (Python 3.11, decoupler 2.1.6). Reads `results/tables/DE_*.csv`
produced by notebook 03.

In [ ]:
import os, re
from pathlib import Path
import numpy as np, pandas as pd
import decoupler as dc
import anndata as ad
import matplotlib.pyplot as plt

# project root: works whether launched from notebooks/ or project root
PROJ = Path(os.environ.get('A04_PROJ', '..')).resolve()
if not (PROJ/'results/tables').exists() and (Path.cwd()/'results/tables').exists():
    PROJ = Path.cwd()
TAB, FIG = PROJ/'results/tables', PROJ/'results/figures'
OUT = PROJ/'results/tf_activity'; OUT.mkdir(parents=True, exist_ok=True)
GTF = PROJ/'data/refs/human_gencode/gencode.v46.primary_assembly.annotation.gtf'
print('project :', PROJ)
print('decoupler:', dc.__version__, '| has op:', hasattr(dc,'op'), '| has mt:', hasattr(dc,'mt'))

## 1. Ensembl → symbol map (from the GENCODE GTF)

In [ ]:
def gtf_symbol_map(gtf):
    d={}; gid=re.compile(r'gene_id "([^".]+)'); gnm=re.compile(r'gene_name "([^"]+)"')
    if not Path(gtf).exists():
        print('   !! GTF missing -> keeping Ensembl IDs'); return d
    with open(gtf) as fh:
        for ln in fh:
            if ln.startswith('#'): continue
            f=ln.split('\t')
            if len(f)>2 and f[2]=='gene':
                a,b=gid.search(ln),gnm.search(ln)
                if a and b: d[a.group(1)]=b.group(1)
    return d
sym = gtf_symbol_map(GTF)
print('symbols mapped:', len(sym))

## 2. CollecTRI regulons (decoupler 2.x `dc.op.collectri`)

In [ ]:
net = dc.op.collectri(organism='human')
# normalise the weight/mor column name across builds
if 'weight' not in net.columns:
    for alt in ('mor','sign_decision'):
        if alt in net.columns:
            net = net.rename(columns={alt:'weight'}); break
print('CollecTRI edges:', len(net), '| cols:', list(net.columns)[:5])
net.head(3)

## 3. Build the contrast × gene statistic matrix

Uses the DESeq2 Wald `stat` (signed, variance-stabilised) per contrast — the
recommended input for ULM. Falls back to signed −log10(padj) if `stat` is absent.

In [ ]:
contrasts = ['Butyrate_10v0_Cd','Butyrate_10v0_Mock','Cd_vs_Mock_0mM','Cd_vs_Mock_10mM']

def load_stat(tag):
    f = TAB/f'DE_{tag}.csv'
    if not f.exists():
        print('   skip (missing):', f.name); return None
    df = pd.read_csv(f, index_col=0)
    s = df['stat'] if 'stat' in df.columns else \
        np.sign(df['log2FoldChange'])*-np.log10(df['padj'].clip(lower=1e-300))
    s.index = [sym.get(re.sub(r'\..*','',g), g) for g in s.index]   # to symbols
    return s[~s.index.duplicated()].dropna()

mats = {}
for t in contrasts:
    s = load_stat(t)
    if s is not None: mats[t] = s
mat = pd.DataFrame(mats).T.fillna(0.0)         # rows=contrasts, cols=genes
print('stat matrix:', mat.shape, '(contrasts x genes)')
mat.iloc[:, :5]

## 4. Restrict regulons to **expressed** genes — the AR-trap fix

A06 ranked AR from network topology even though we now know it is unexpressed in the
organoid. Here we drop every regulon edge whose target is not in our measured matrix,
so only TFs acting on genes that are actually transcribed can score.

In [ ]:
net_expr = net[net['target'].isin(mat.columns)].copy()
print(f'edges kept (expressed targets): {len(net_expr)} / {len(net)}')
print('TFs with >=5 expressed targets:',
      (net_expr.groupby('source').size() >= 5).sum())

## 5. ULM activity (decoupler 2.x writes into `AnnData.obsm`)

In [ ]:
adata = ad.AnnData(mat)                      # obs = contrasts, var = genes
dc.mt.ulm(data=adata, net=net_expr, tmin=5, verbose=True)

# 2.x stores activities + FDR in obsm
acts  = adata.obsm['score_ulm'].copy()
pvals = adata.obsm.get('padj_ulm', adata.obsm.get('pvals_ulm')).copy()
acts.index = mat.index; pvals.index = mat.index
acts.to_csv(OUT/'tf_activity_scores.csv')
pvals.to_csv(OUT/'tf_activity_pvals.csv')
print('TFs scored:', acts.shape[1])

## 6. Candidate controllers — TFs DOWN with butyrate in Cd

The headline question. A shared controller of the coupled module should *lose* activity
as butyrate rises in the infected arm.

In [ ]:
key = 'Butyrate_10v0_Cd'
a = acts.loc[key].sort_values()
p = pvals.loc[key].reindex(a.index)
res = pd.DataFrame({'activity': a, 'padj': p})
res.to_csv(OUT/'TF_butyrate_in_Cd_ranked.csv')

print('#### TFs DOWN with butyrate in Cd (candidate controllers) ####')
print(res.head(15).round(3).to_string())
print('\n#### TFs UP with butyrate in Cd ####')
print(res.tail(8).round(3).to_string())

## 7. The shared-controller test: down-with-butyrate **and** up-with-infection

A TF that is induced by infection (Cd_vs_Mock_0mM > 0) *and* suppressed by butyrate in
Cd (< 0) is the signature of a controller of the coupled, butyrate-reversible module.

In [ ]:
inf = acts.loc['Cd_vs_Mock_0mM']
but = acts.loc['Butyrate_10v0_Cd']
shared = pd.DataFrame({'inf_0mM': inf, 'but_10v0_Cd': but,
                       'padj_but': pvals.loc['Butyrate_10v0_Cd']})
# candidate = up with infection, down with butyrate
cand = shared[(shared.inf_0mM > 0) & (shared.but_10v0_Cd < 0)].copy()
cand['score'] = cand.inf_0mM - cand.but_10v0_Cd          # bigger = stronger reversal
cand = cand.sort_values('score', ascending=False)
cand.to_csv(OUT/'TF_shared_controller_candidates.csv')
print('candidate shared controllers (up-inf & down-but):', len(cand))
print(cand.head(15).round(3).to_string())

print('\n#### targeted readouts ####')
for tf in ['NFKB1','RELA','RELB','SP1','SREBF2','XBP1','ATF4','ATF6',
           'CREB3L1','CREB3L2','NFE2L2','NFYA','CEBPB','FOXA1','FOXA2','HNF4A']:
    if tf in acts.columns:
        print(f'   {tf:8s} but10v0_Cd={but[tf]:+.2f}  inf@0mM={inf[tf]:+.2f}')

## 8. Figures — candidate scatter + cross-contrast heatmap

In [ ]:
# (A) infection vs butyrate TF-activity scatter; candidates land top-left
fig, ax = plt.subplots(figsize=(5.5,5))
ax.axhline(0, lw=.5, c='grey'); ax.axvline(0, lw=.5, c='grey')
ax.scatter(shared.inf_0mM, shared.but_10v0_Cd, s=10, c='#bbbbbb')
top = cand.head(12)
ax.scatter(top.inf_0mM, top.but_10v0_Cd, s=28, c='#cb181d')
for tf,row in top.iterrows():
    ax.annotate(tf, (row.inf_0mM, row.but_10v0_Cd), fontsize=7)
ax.set_xlabel('TF activity — infection @0mM'); ax.set_ylabel('TF activity — butyrate 10v0 in Cd')
ax.set_title('Candidate controllers: up with infection, down with butyrate')
plt.tight_layout(); fig.savefig(FIG/'tf_candidate_scatter.png', dpi=130); plt.show()

# (B) heatmap of top movers across all contrasts
topmov = acts.loc[key].abs().sort_values(ascending=False).head(25).index
H = acts[topmov].T
fig, ax = plt.subplots(figsize=(6, .32*len(topmov)+1))
im = ax.imshow(H.values, cmap='RdBu_r', vmin=-4, vmax=4, aspect='auto')
ax.set_xticks(range(H.shape[1])); ax.set_xticklabels(H.columns, rotation=45, ha='right', fontsize=7)
ax.set_yticks(range(len(topmov))); ax.set_yticklabels(topmov, fontsize=7)
fig.colorbar(im, label='TF activity (ULM)'); ax.set_title('TF activity across A04 contrasts')
plt.tight_layout(); fig.savefig(FIG/'tf_activity_heatmap.png', dpi=130); plt.show()

## 9. Readout

- **NFKB1 / RELA down with butyrate** → confirmatory (the chemokine arm). Expected.
- **A stress/chromatin or colon-identity TF** (CEBPB, NFE2L2, ATF, HNF4A/FOXA) appearing
  in the candidate list → a genuine lead for the chaperone-arm controller.
- **No single TF dominating the chaperone genes** → also a valid result: the chaperone
  induction is distributed / chromatin-level (consistent with the butyrate–HDAC framing),
  to be written as *"no single CollecTRI regulator accounts for the chaperone arm."*

Outputs: `results/tf_activity/TF_butyrate_in_Cd_ranked.csv`,
`TF_shared_controller_candidates.csv`, and figures in `results/figures/`.